# 🧠 NeuraQuiz AI — Complete GPU Pipeline
**Intelligent Reading Comprehension & Quiz Generation System**

This notebook runs the entire training pipeline on your **NVIDIA RTX 5070 Ti** GPU.

| Step | Description | Time |
|------|-------------|------|
| 1 | Preprocessing (TF-IDF & One-Hot Feature Eng.) | ~6 min |
| 2 | Model A Training (9 models) | ~6 min |
| 3 | Model B Training (Distractor + SBERT) | ~2 min |
| 4 | Launch FastAPI Backend | Background |
| 5 | Launch React Frontend | Background |

---
## Step 1: Preprocessing (GPU-Accelerated Feature Engineering)
Generates TF-IDF features, One-Hot Baseline features, and GPU-computed cosine similarities.

In [1]:
!python src/preprocessing.py

[WARN] Environment check skipped (check_pkgs.py not found).
[GPU] Preprocessing active on: NVIDIA GeForce RTX 5070 Ti Laptop GPU
[WARN] Detected identical file sizes for train/val/test. Performing manual split (80/10/10)...
[INFO] Dataset split -> Train: 70292 | Val: 8787 | Test: 8787
[INFO] TF-IDF vocabulary size: 15000
[INFO] Saved -> C:\Users\dotsm\Desktop\AI LAB Project\race_rc_project\data\processed\tfidf_vectorizer.pkl
[INFO] One-Hot vocabulary size: 15000
[INFO] Saved -> C:\Users\dotsm\Desktop\AI LAB Project\race_rc_project\data\processed\onehot_vectorizer.pkl
[WARN] Neural Networks banned by instructor. Skipping SBERT features.

[INFO] Building verification features for TRAIN (this may take a while)...
[INFO] Vectorizing articles, questions, and options (TF-IDF)...
[INFO] Computing TF-IDF similarity features on GPU...
[INFO]   Computing word overlap for option A...
[INFO]   Computing word overlap for option B...
[INFO]   Computing word overlap for option C...
[INFO]   Computing

---
## Step 2: Train Model A — Answer Verification
Trains: XGBoost, PyTorch MLP, K-Means, Logistic Regression, Random Forest, SVM, Naive Bayes, Label Propagation, and Soft-Voting Ensemble.

In [2]:
!python src/model_a_train.py

[WARN] Environment check skipped (check_pkgs.py not found).
[WARN] Environment check skipped (check_pkgs.py not found).
[GPU] Preprocessing active on: NVIDIA GeForce RTX 5070 Ti Laptop GPU
[GPU] ACTIVE: NVIDIA GeForce RTX 5070 Ti Laptop GPU
[GPU] VRAM: 12.8 GB
[GPU] Enforcement: All compatible operations will run on CUDA device 0.
[INFO] Loaded <- C:\Users\dotsm\Desktop\AI LAB Project\race_rc_project\data\processed\train_verification_features.pkl
[INFO] Loaded <- C:\Users\dotsm\Desktop\AI LAB Project\race_rc_project\data\processed\val_verification_features.pkl
[INFO] Loaded <- C:\Users\dotsm\Desktop\AI LAB Project\race_rc_project\data\processed\test_verification_features.pkl

  [GPU] Training: XGBoost (CUDA)
  Class imbalance: 210876:70292 -> scale_pos_weight=3.00
[0]	validation_0-logloss:0.69270
[50]	validation_0-logloss:0.68283
[100]	validation_0-logloss:0.68038
[150]	validation_0-logloss:0.67926
[200]	validation_0-logloss:0.67824
[250]	validation_0-logloss:0.67748
[300]	validation_0

C:\Users\dotsm\AppData\Roaming\Python\Python314\site-packages\xgboost\core.py:751: UserWarning: [22:59:12] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


---
## Step 3: Train Model B — Distractor Generator
Trains the XGBoost distractor ranker and prepares SBERT.

In [3]:
!python src/model_b_train.py

[WARN] Environment check skipped (check_pkgs.py not found).
[WARN] Environment check skipped (check_pkgs.py not found).
[GPU] Preprocessing active on: NVIDIA GeForce RTX 5070 Ti Laptop GPU
[GPU] ACTIVE: NVIDIA GeForce RTX 5070 Ti Laptop GPU
[GPU] Enforcement: SBERT and XGBoost will run exclusively on CUDA device 0.
[WARN] Detected identical file sizes for train/val/test. Performing manual split (80/10/10)...
[INFO] Dataset split -> Train: 70292 | Val: 8787 | Test: 8787
[INFO] Loaded <- C:\Users\dotsm\Desktop\AI LAB Project\race_rc_project\data\processed\tfidf_vectorizer.pkl

[PHASE] Building distractor training data...
[INFO] Distractor data: X=(60000, 5) y=(60000,) distractor_rate=0.7500
[INFO] Distractor data: X=(12000, 5) y=(12000,) distractor_rate=0.7500

  [GPU] Training: Distractor Ranker (XGBoost CUDA)
  Class imbalance: 15000:45000 -> scale_pos_weight=0.333
[0]	validation_0-logloss:0.59849
[50]	validation_0-logloss:0.00975
[89]	validation_0-logloss:0.00842
  Time: 0.5s
  Accura

C:\Users\dotsm\AppData\Roaming\Python\Python314\site-packages\xgboost\core.py:751: UserWarning: [23:05:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


---
## Step 4: Launch Backend & Frontend Servers
Run this cell to start both the FastAPI backend and the React frontend in the background. They will stay running until you shut down the kernel.

In [4]:
import subprocess
import time
import os

print("🚀 Starting FastAPI Backend on port 8000...")
backend_process = subprocess.Popen(
    ["python", "-m", "uvicorn", "main:app", "--port", "8000"],
    cwd=os.path.join(os.getcwd(), "api"),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    shell=True
)

print("🚀 Starting React Frontend on port 5173...")
frontend_process = subprocess.Popen(
    ["npm", "run", "dev"],
    cwd=os.path.join(os.getcwd(), "frontend"),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    shell=True
)

time.sleep(3)
print("\n✅ Servers are live!")
print("🌐 Open your browser to: http://localhost:5173/")

🚀 Starting FastAPI Backend on port 8000...
🚀 Starting React Frontend on port 5173...



✅ Servers are live!
🌐 Open your browser to: http://localhost:5173/
